# Scraping Elecciones Colombia — 8 de marzo de 2026

Extracción estructurada de datos electorales desde el portal de resultados de la Registraduría Nacional.

**Fuente**: `https://resultados.registraduria.gov.co`

> **Nota técnica**: La cookie `cf_clearance` (protección Cloudflare) puede vencer en horas o días. Si recibes un error `403 Forbidden`, copia una nueva cookie desde el navegador y actualiza el archivo `config.json` en la raíz del proyecto.

In [1]:
import json
import os
import requests
import pandas as pd

# Cargar configuración centralizada (cookies, headers, URLs)
CONFIG_PATH = os.path.join('..', '..', 'config.json')
with open(CONFIG_PATH) as f:
    config = json.load(f)

URL = config['nomenclator_url']
cookies = config['cookies']
headers = config['headers']

response = requests.get(URL, cookies=cookies, headers=headers)

if response.status_code == 200:
    data = response.json()
    print(f"✅ nomenclator.json cargado ({len(data['elec'])} elecciones, {len(data['partidos'])} partidos)")
else:
    print("❌ Request failed with status code:", response.status_code)
    print("   → Verifica que cf_clearance en config.json sea válido")

✅ nomenclator.json cargado (4 elecciones, 348 partidos)


# Paso 1: Entender la estructura del JSON

El archivo `nomenclator.json` es el **diccionario maestro** de las elecciones. La Registraduría lo usa para que los archivos de resultados sean livianos: en vez de enviar nombres completos, envía códigos numéricos. Este JSON nos permite "traducir" esos códigos a nombres reales.

**Llaves principales del JSON:**
| Llave | Descripción |
|-------|-------------|
| `ver` | Versión del archivo |
| `y` | Año electoral (2026) |
| `elec` | Lista de **elecciones/corporaciones** (Senado, Cámara, Consultas, CITREP) con sus circunscripciones |
| `levels` | **Niveles** de la jerarquía geográfica (Colombia → Departamento → Municipio → Zona → Comuna → Puesto → Mesa) |
| `amb` | **Ámbitos** (División Político-Administrativa / Divipol). Contiene todos los lugares organizados jerárquicamente |
| `partidos` | Lista de **partidos políticos** con código, nombre y color |

A continuación construiremos un DataFrame (tabla de dimensión) para cada entidad.

In [2]:
# Verificamos las llaves principales del JSON
print("Llaves principales del nomenclator:")
for key in data.keys():
    val = data[key]
    tipo = type(val).__name__
    if isinstance(val, list):
        print(f"  '{key}' -> {tipo}, {len(val)} elementos")
    elif isinstance(val, dict):
        print(f"  '{key}' -> {tipo}")
    else:
        print(f"  '{key}' -> {tipo}: {val}")

Llaves principales del nomenclator:
  'ver' -> int: 1
  'y' -> str: 2026
  'elec' -> list, 4 elementos
  'levels' -> list, 7 elementos
  'amb' -> list, 4 elementos
  'partidos' -> list, 348 elementos


## Paso 2: Tabla de dimensión — Elecciones y Circunscripciones (`dim_elecciones`)

Cada elección (Senado, Cámara, etc.) tiene una o más **circunscripciones** (también llamadas "cámaras" en el JSON, campo `cam`). Por ejemplo, el Senado tiene circunscripción Nacional e Indígena, mientras que Cámara tiene Territorial Departamental, Indígena y Afro.

Creamos una tabla plana donde cada fila es una combinación **elección + circunscripción**.

In [3]:
# Construimos dim_elecciones: una fila por cada combinación elección + circunscripción
filas_elecciones = []

for eleccion in data['elec']:
    for cam in eleccion['cam']:
        filas_elecciones.append({
            'elec_id': eleccion['i'],          # Índice interno de la elección
            'elec_codigo': eleccion['elec'],    # Código de la elección usado en los archivos de votos
            'elec_orden': eleccion['ord'],      # Orden de presentación
            'elec_sigla': eleccion['sigla'],    # Sigla (SE, CA, CN, CT)
            'elec_nombre': eleccion['n'],       # Nombre completo de la elección
            'elec_color': eleccion['c'],        # Color principal
            'elec_color_sec': eleccion['cs'],   # Color secundario
            'cam_id': cam['i'],                 # Índice de la circunscripción
            'cam_nombre': cam['n'],             # Nombre de la circunscripción
            'cam_color': cam['c'],              # Color de la circunscripción
        })

dim_elecciones = pd.DataFrame(filas_elecciones)
print(f"dim_elecciones: {len(dim_elecciones)} filas")
dim_elecciones

dim_elecciones: 7 filas


,elec_id,elec_codigo,elec_orden,elec_sigla,elec_nombre,elec_color,elec_color_sec,cam_id,cam_nombre,cam_color
0,0,1,1,SE,SENADO,#1C283D,#47658F,0,NACIONAL,#FFC53D
1,0,1,1,SE,SENADO,#1C283D,#47658F,4,INDIGENAS,#004F9A
2,1,2,2,CA,CAMARA,#1C283D,#919662,1,TERRITORIAL DEPARTAMENTAL,#FFC53D
3,1,2,2,CA,CAMARA,#1C283D,#919662,4,INDIGENAS,#004F9A
4,1,2,2,CA,CAMARA,#1C283D,#919662,5,AFRO-DESCENDIENTES,#EC0000
5,2,6,3,CN,CONSULTAS INTERPARTIDISTAS,#1C283D,#64a70b,0,NACIONAL,#FFC53D
6,3,7,4,CT,CITREP,#1C283D,#925862,9,CITREP,#A2789C


## Paso 3: Tabla de dimensión — Niveles Geográficos (`dim_niveles`)

La jerarquía geográfica de la Registraduría tiene 7 niveles. Esta tabla es pequeña pero crucial para entender qué nivel (`l`) corresponde a qué tipo de entidad territorial.

`1: COLOMBIA → 2: DEPARTAMENTO → 3: MUNICIPIO → 4: ZONA → 5: COMUNA → 6: PUESTO → 7: MESA`

In [5]:
# Construimos dim_niveles: directo del array 'levels'
dim_niveles = pd.DataFrame(data['levels'])
dim_niveles.columns = ['nivel_id', 'nivel_nombre']

print(f"dim_niveles: {len(dim_niveles)} filas")
dim_niveles

dim_niveles: 7 filas


,nivel_id,nivel_nombre
0,1,COLOMBIA
1,2,DEPARTAMENTO
2,3,MUNICIPIO
3,4,ZONA
4,5,COMUNA
5,6,PUESTO
6,7,MESA


## Paso 4: Tabla de dimensión — Partidos Políticos (`dim_partidos`)

Cada partido tiene un código (`codpar`) que se usa en los archivos de resultados de votos. Esta tabla nos permite traducir ese código al nombre completo del partido y su color oficial.

Hay **348 registros** que incluyen partidos, movimientos, coaliciones y categorías especiales como "Votos en blanco, nulos y no marcados".

In [7]:
# Construimos dim_partidos: directo del array 'partidos'
dim_partidos = pd.DataFrame(data['partidos'])

# Renombramos columnas a nombres descriptivos
dim_partidos.rename(columns={
    'codpar': 'partido_codigo',
    'nombre': 'partido_nombre',
    'color': 'partido_color',
    'i': 'partido_indice',
    's': 'partido_slug',
}, inplace=True)

print(f"dim_partidos: {len(dim_partidos)} filas")
dim_partidos.head(10)

dim_partidos: 348 filas


,partido_codigo,partido_nombre,partido_color,partido_indice,partido_slug
0,0,"VOTOS EN BLANCO, NULOS Y NO MARCADOS",#5A7CCC,1,"VOTOS-EN-BLANCO,-NULOS-Y-NO-MARCADOS"
1,1,PARTIDO LIBERAL COLOMBIANO,#E30716,2,PARTIDO-LIBERAL-COLOMBIANO
2,2,PARTIDO CONSERVADOR COLOMBIANO,#0867B1,3,PARTIDO-CONSERVADOR-COLOMBIANO
3,3,PARTIDO CAMBIO RADICAL,#E3051C,4,PARTIDO-CAMBIO-RADICAL
4,4,PARTIDO ALIANZA VERDE,#007C34,5,PARTIDO-ALIANZA-VERDE
5,5,"MOVIMIENTO AUTORIDADES INDÍGENAS DE COLOMBIA ""...",#C05B16,6,"MOVIMIENTO-AUTORIDADES-INDIGENAS-DE-COLOMBIA-""..."
6,6,"PARTIDO ALIANZA SOCIAL INDEPENDIENTE ""ASI""",#F7C412,7,"PARTIDO-ALIANZA-SOCIAL-INDEPENDIENTE-""ASI"""
7,7,PARTIDO POLÍTICO MIRA,#06539C,8,PARTIDO-POLITICO-MIRA
8,8,PARTIDO DE LA UNIÓN POR LA GENTE - PARTIDO DE ...,#48AB38,9,PARTIDO-DE-LA-UNION-POR-LA-GENTE---PARTIDO-DE-...
9,11,PARTIDO CENTRO DEMOCRÁTICO,#1E477D,10,PARTIDO-CENTRO-DEMOCRATICO


## Paso 5: Tabla de dimensión — Ámbitos Geográficos / Divipol (`dim_ambitos`)

Esta es la tabla más importante y compleja. El array `amb` contiene la **División Político-Administrativa (Divipol)** organizada por elección. Cada ámbito tiene:

| Campo | Significado |
|-------|-------------|
| `i` | Índice interno (posición en el array) |
| `n` | Nombre del lugar |
| `co` | Código Divipol de la Registraduría |
| `s` | Slug (nombre para URL) |
| `l` | Nivel jerárquico (1=Colombia, 2=Departamento, 3=Municipio) |
| `p` | **Padres**: lista de índices que apuntan al nivel superior |
| `h` | **Hijos**: lista de índices que apuntan al nivel inferior |
| `r` | Array de resoluciones/referencias internas |

Vamos a extraer una tabla plana de ámbitos para cada elección, resolviendo la relación padre para saber a qué departamento pertenece cada municipio.

In [8]:
# Primero exploramos cuántos ámbitos hay por elección y por nivel
for grupo in data['amb']:
    elec_id = grupo['elec']
    ambitos = grupo['ambitos']
    
    # Conteo por nivel
    conteo = {}
    for a in ambitos:
        nivel = a['l']
        conteo[nivel] = conteo.get(nivel, 0) + 1
    
    print(f"Elección {elec_id}: {len(ambitos)} ámbitos totales")
    for nivel in sorted(conteo.keys()):
        nombre_nivel = next((l['n'] for l in data['levels'] if l['l'] == nivel), '?')
        print(f"  Nivel {nivel} ({nombre_nivel}): {conteo[nivel]}")
    print()

Elección 1: 1224 ámbitos totales
  Nivel 1 (COLOMBIA): 1
  Nivel 2 (DEPARTAMENTO): 34
  Nivel 3 (MUNICIPIO): 1189

Elección 2: 1224 ámbitos totales
  Nivel 1 (COLOMBIA): 1
  Nivel 2 (DEPARTAMENTO): 34
  Nivel 3 (MUNICIPIO): 1189

Elección 6: 1224 ámbitos totales
  Nivel 1 (COLOMBIA): 1
  Nivel 2 (DEPARTAMENTO): 34
  Nivel 3 (MUNICIPIO): 1189

Elección 7: 185 ámbitos totales
  Nivel 1 (COLOMBIA): 1
  Nivel 2 (DEPARTAMENTO): 16
  Nivel 3 (MUNICIPIO): 168



In [9]:
# Verificamos si las elecciones 1, 2 y 6 comparten los mismos ámbitos
# Comparamos los códigos (co) y nombres (n) de los ámbitos
ambitos_por_elec = {}
for grupo in data['amb']:
    codigos = set((a['co'], a['n'], a['l']) for a in grupo['ambitos'])
    ambitos_por_elec[grupo['elec']] = codigos

# Comparar elección 1 vs 2
iguales_1_2 = ambitos_por_elec[1] == ambitos_por_elec[2]
iguales_1_6 = ambitos_por_elec[1] == ambitos_por_elec[6]
print(f"¿Elecciones 1 y 2 tienen los mismos ámbitos? {iguales_1_2}")
print(f"¿Elecciones 1 y 6 tienen los mismos ámbitos? {iguales_1_6}")
print(f"Elección 7 (CITREP) tiene un subconjunto de {len(ambitos_por_elec[7])} ámbitos")

¿Elecciones 1 y 2 tienen los mismos ámbitos? True
¿Elecciones 1 y 6 tienen los mismos ámbitos? True
Elección 7 (CITREP) tiene un subconjunto de 185 ámbitos


### Construcción de `dim_ambitos`

Las elecciones 1 (Senado), 2 (Cámara) y 6 (Consultas) comparten **exactamente los mismos 1224 ámbitos**. La elección 7 (CITREP) usa un subconjunto de 185.

Tomamos los ámbitos de la elección 1 como base y resolvemos la relación padre-hijo para crear una tabla donde cada municipio tenga su departamento asociado.

**Lógica de resolución del padre**: El campo `p` de cada ámbito contiene una lista de objetos `{l: nivel, p: [índices]}`. Para un municipio (nivel 3), su padre es el departamento (nivel 2) cuyo índice está en `p[0].p[0]`.

In [10]:
# Tomamos los ámbitos de la elección 1 (Senado) como base completa
ambitos_base = data['amb'][0]['ambitos']  # elec=1

# Paso 1: Crear un diccionario de lookup por índice para resolver padres
lookup = {a['i']: a for a in ambitos_base}

# Paso 2: Construir filas planas resolviendo la jerarquía
filas_ambitos = []

for a in ambitos_base:
    fila = {
        'ambito_indice': a['i'],            # Índice interno (posición en el array)
        'ambito_nombre': a['n'],             # Nombre del lugar
        'ambito_codigo': a['co'],            # Código Divipol
        'ambito_slug': a.get('s', ''),       # Slug para URL
        'nivel_id': a['l'],                  # Nivel jerárquico
    }
    
    # Resolver el padre inmediato (si existe)
    # El campo 'p' es una lista de relaciones: [{l: nivel_padre, p: [indices_padre]}]
    padre_indice = None
    if a['p']:
        # Tomamos la primera relación padre
        padre_rel = a['p'][0]
        if padre_rel['p']:
            padre_indice = padre_rel['p'][0]
    
    fila['padre_indice'] = padre_indice
    
    # Si es un municipio (nivel 3), resolver el nombre del departamento padre
    if a['l'] == 3 and padre_indice is not None:
        padre = lookup.get(padre_indice, {})
        fila['departamento_nombre'] = padre.get('n', '')
        fila['departamento_codigo'] = padre.get('co', '')
    elif a['l'] == 2:
        # El departamento es su propio departamento
        fila['departamento_nombre'] = a['n']
        fila['departamento_codigo'] = a['co']
    else:
        fila['departamento_nombre'] = None
        fila['departamento_codigo'] = None
    
    filas_ambitos.append(fila)

dim_ambitos = pd.DataFrame(filas_ambitos)
print(f"dim_ambitos: {len(dim_ambitos)} filas")
print("\nDistribución por nivel:")
print(dim_ambitos.groupby('nivel_id').size().reset_index(name='cantidad'))
dim_ambitos.head(10)

dim_ambitos: 1224 filas

Distribución por nivel:
   nivel_id  cantidad
0         1         1
1         2        34
2         3      1189


,ambito_indice,ambito_nombre,ambito_codigo,ambito_slug,nivel_id,padre_indice,departamento_nombre,departamento_codigo
0,0,COLOMBIA,00,COLOMBIA,1,NaN,NaN,NaN
1,1,AMAZONAS,6000,AMAZONAS,2,0.0,AMAZONAS,6000
2,2,ANTIOQUIA,0100,ANTIOQUIA,2,0.0,ANTIOQUIA,0100
3,3,ARAUCA,4000,ARAUCA,2,0.0,ARAUCA,4000
4,4,ATLANTICO,0300,ATLANTICO,2,0.0,ATLANTICO,0300
5,5,BOGOTA D.C.,1600,BOGOTA-D.C.,2,0.0,BOGOTA D.C.,1600
6,6,BOLIVAR,0500,BOLIVAR,2,0.0,BOLIVAR,0500
7,7,BOYACA,0700,BOYACA,2,0.0,BOYACA,0700
8,8,CALDAS,0900,CALDAS,2,0.0,CALDAS,0900
9,9,CAQUETA,4400,CAQUETA,2,0.0,CAQUETA,4400


## Paso 6: Tablas de conveniencia — Departamentos y Municipios

A partir de `dim_ambitos` creamos dos vistas filtradas que son las más útiles para análisis:

- **`dim_departamentos`**: Solo los 34 departamentos (nivel 2)
- **`dim_municipios`**: Los 1189 municipios (nivel 3) con su departamento ya resuelto

También creamos `dim_ambitos_citrep` para los 185 ámbitos específicos de la elección CITREP (Circunscripción Transitoria Especial de Paz).

In [11]:
# --- dim_departamentos: solo nivel 2 ---
dim_departamentos = dim_ambitos[dim_ambitos['nivel_id'] == 2][
    ['ambito_indice', 'ambito_nombre', 'ambito_codigo']
].rename(columns={
    'ambito_indice': 'depto_indice',
    'ambito_nombre': 'depto_nombre',
    'ambito_codigo': 'depto_codigo',
}).reset_index(drop=True)

print(f"dim_departamentos: {len(dim_departamentos)} filas")
dim_departamentos

dim_departamentos: 34 filas


,depto_indice,depto_nombre,depto_codigo
0,1,AMAZONAS,6000
1,2,ANTIOQUIA,0100
2,3,ARAUCA,4000
3,4,ATLANTICO,0300
4,5,BOGOTA D.C.,1600
5,6,BOLIVAR,0500
6,7,BOYACA,0700
7,8,CALDAS,0900
8,9,CAQUETA,4400
9,10,CASANARE,4600


In [12]:
# --- dim_municipios: solo nivel 3, con departamento resuelto ---
dim_municipios = dim_ambitos[dim_ambitos['nivel_id'] == 3][
    ['ambito_indice', 'ambito_nombre', 'ambito_codigo', 'departamento_nombre', 'departamento_codigo']
].rename(columns={
    'ambito_indice': 'mpio_indice',
    'ambito_nombre': 'mpio_nombre',
    'ambito_codigo': 'mpio_codigo',
    'departamento_nombre': 'depto_nombre',
    'departamento_codigo': 'depto_codigo',
}).reset_index(drop=True)

print(f"dim_municipios: {len(dim_municipios)} filas")
dim_municipios.head(10)

dim_municipios: 1189 filas


,mpio_indice,mpio_nombre,mpio_codigo,depto_nombre,depto_codigo
0,35,ABEJORRAL,0100004,ANTIOQUIA,0100
1,36,ABREGO,2500004,NORTE DE SAN,2500
2,37,ABRIAQUI,0100007,ANTIOQUIA,0100
3,38,ACACIAS,5200005,META,5200
4,39,ACANDI,1700004,CHOCO,1700
5,40,ACEVEDO,1900004,HUILA,1900
6,41,ACHI,0500004,BOLIVAR,0500
7,42,AGRADO,1900007,HUILA,1900
8,43,AGUA DE DIOS,1500004,CUNDINAMARCA,1500
9,44,AGUACHICA,1200075,CESAR,1200


In [13]:
# --- dim_ambitos_citrep: ámbitos específicos de CITREP (elección 7) ---
# CITREP = Circunscripción Transitoria Especial de Paz
ambitos_citrep = data['amb'][3]['ambitos']  # elec=7
lookup_citrep = {a['i']: a for a in ambitos_citrep}

filas_citrep = []
for a in ambitos_citrep:
    padre_indice = None
    if a['p']:
        padre_rel = a['p'][0]
        if padre_rel['p']:
            padre_indice = padre_rel['p'][0]
    
    fila = {
        'ambito_indice': a['i'],
        'ambito_nombre': a['n'],
        'ambito_codigo': a['co'],
        'nivel_id': a['l'],
        'padre_indice': padre_indice,
    }
    
    # Resolver departamento para municipios
    if a['l'] == 3 and padre_indice is not None:
        padre = lookup_citrep.get(padre_indice, {})
        fila['departamento_nombre'] = padre.get('n', '')
    elif a['l'] == 2:
        fila['departamento_nombre'] = a['n']
    else:
        fila['departamento_nombre'] = None
    
    filas_citrep.append(fila)

dim_ambitos_citrep = pd.DataFrame(filas_citrep)
print(f"dim_ambitos_citrep: {len(dim_ambitos_citrep)} filas")
print(f"  Departamentos CITREP: {len(dim_ambitos_citrep[dim_ambitos_citrep['nivel_id'] == 2])}")
print(f"  Municipios CITREP: {len(dim_ambitos_citrep[dim_ambitos_citrep['nivel_id'] == 3])}")
dim_ambitos_citrep[dim_ambitos_citrep['nivel_id'] == 3].head(10)

dim_ambitos_citrep: 185 filas
  Departamentos CITREP: 16
  Municipios CITREP: 168


,ambito_indice,ambito_nombre,ambito_codigo,nivel_id,padre_indice,departamento_nombre
17,17,ACANDI,1706004,3,13.0,CIRCUNSCRIPCIÓN 6 (CHOCÓ - ANT
18,18,AGUSTIN CODAZZI,1212150,3,4.0,CIRCUNSCRIPCIÓN 12 (CESAR - LA
19,19,ALBANIA,4405002,3,12.0,CIRCUNSCRIPCIÓN 5 (CAQUETÁ - H
20,20,ALGECIRAS,1905013,3,12.0,CIRCUNSCRIPCIÓN 5 (CAQUETÁ - H
21,21,AMALFI,0103016,3,10.0,CIRCUNSCRIPCIÓN 3 ANTIOQUIA
22,22,ANORI,0103028,3,10.0,CIRCUNSCRIPCIÓN 3 ANTIOQUIA
23,23,APARTADO,0116035,3,8.0,CIRCUNSCRIPCIÓN 16 ANTIOQUIA
24,24,ARACATACA,2112010,3,4.0,CIRCUNSCRIPCIÓN 12 (CESAR - LA
25,25,ARAUQUITA,4002010,3,9.0,CIRCUNSCRIPCIÓN 2 ARAUCA
26,26,ARENAL,0513005,3,5.0,CIRCUNSCRIPCIÓN 13 (BOLÍVAR -


## Paso 7: Resumen de tablas de dimensión creadas

Todas las tablas de dimensión están listas para ser usadas como lookup al procesar los archivos de resultados de votos.

| DataFrame | Filas | Descripción |
|-----------|-------|-------------|
| `dim_elecciones` | 7 | Elecciones + circunscripciones (Senado Nacional, Senado Indígena, Cámara Territorial, etc.) |
| `dim_niveles` | 7 | Jerarquía geográfica (Colombia → Depto → Municipio → Zona → Comuna → Puesto → Mesa) |
| `dim_partidos` | 348 | Partidos políticos con código, nombre y color |
| `dim_ambitos` | 1,224 | Todos los ámbitos geográficos (país + deptos + municipios) con jerarquía resuelta |
| `dim_departamentos` | 34 | Vista filtrada: solo departamentos |
| `dim_municipios` | 1,189 | Vista filtrada: solo municipios con su departamento |
| `dim_ambitos_citrep` | 185 | Ámbitos específicos de las Circunscripciones Transitorias Especiales de Paz |

**Uso**: Cuando los archivos de votos traigan un código como `"0100004"`, se puede buscar en `dim_ambitos` para saber que es **ABEJORRAL, ANTIOQUIA**. Igualmente, un `codpar = "1"` se traduce a **PARTIDO LIBERAL COLOMBIANO** usando `dim_partidos`.

In [14]:
# Resumen final: todas las tablas de dimensión en memoria
tablas = {
    'dim_elecciones': dim_elecciones,
    'dim_niveles': dim_niveles,
    'dim_partidos': dim_partidos,
    'dim_ambitos': dim_ambitos,
    'dim_departamentos': dim_departamentos,
    'dim_municipios': dim_municipios,
    'dim_ambitos_citrep': dim_ambitos_citrep,
}

print("Tablas de dimensión disponibles:")
print("-" * 50)
for nombre, df in tablas.items():
    print(f"  {nombre:25s} -> {len(df):>5} filas, {len(df.columns)} columnas")
print("-" * 50)
print(f"Total: {len(tablas)} tablas creadas")

Tablas de dimensión disponibles:
--------------------------------------------------
  dim_elecciones            ->     7 filas, 10 columnas
  dim_niveles               ->     7 filas, 2 columnas
  dim_partidos              ->   348 filas, 5 columnas
  dim_ambitos               ->  1224 filas, 8 columnas
  dim_departamentos         ->    34 filas, 3 columnas
  dim_municipios            ->  1189 filas, 5 columnas
  dim_ambitos_citrep        ->   185 filas, 6 columnas
--------------------------------------------------
Total: 7 tablas creadas


# Data Granular — Consultas Interpartidistas (CN)

La Registraduría expone los resultados de votación en endpoints JSON con el patrón:

```
https://resultados.registraduria.gov.co/json/ACT/{SIGLA}/{AMBITO}.json
```

| Parámetro | Descripción | Ejemplo |
|-----------|-------------|---------|
| `SIGLA` | Sigla de la elección (de `dim_elecciones`) | `SE`, `CA`, `CN`, `CT` |
| `AMBITO` | Código geográfico (de `dim_ambitos`) | `00` (nacional), `0100` (Antioquia), `0100004` (Abejorral) |

Cada response contiene:
- **`totales`**: Resumen de mesas, votantes, abstención, nulos
- **`camaras[].partotabla`**: Votos por partido y candidato
- **`camaras[].mapagan`**: Partido ganador por cada sub-ámbito geográfico

En esta sección consultamos **Consultas Interpartidistas (CN)** a la mayor granularidad posible: **nivel municipio**.

### Estrategia de scraping

1. **34 departamentos** (`dim_departamentos`): Cada response trae `partotabla` (votos agregados por candidato) y `mapagan` (ganador por municipio). Con solo 34 requests obtenemos buena cobertura.
2. **1,189 municipios** (`dim_municipios`): Para obtener votos **de cada candidato por municipio** se requiere consultar cada municipio individualmente.

In [15]:
import time

# === Scraping de los 34 departamentos para Consultas (CN) ===
BASE_URL = "https://resultados.registraduria.gov.co/json/ACT/CN"

# Usamos dim_departamentos del nomenclator
codigos_deptos = dim_departamentos['depto_codigo'].tolist()
nombres_deptos = dim_departamentos['depto_nombre'].tolist()

# Almacenar respuestas crudas
responses_cn = {}
errores = []

print(f"Descargando datos de {len(codigos_deptos)} departamentos...")
for i, (codigo, nombre) in enumerate(zip(codigos_deptos, nombres_deptos)):
    url = f"{BASE_URL}/{codigo}.json"
    r = requests.get(url, cookies=cookies, headers=headers)
    
    if r.status_code == 200:
        responses_cn[codigo] = r.json()
        print(f"  [{i+1:2d}/34] ✅ {nombre} ({codigo})")
    else:
        errores.append((codigo, nombre, r.status_code))
        print(f"  [{i+1:2d}/34] ❌ {nombre} ({codigo}) -> HTTP {r.status_code}")
    
    time.sleep(0.3)  # Respetar el servidor

print(f"\nCompletado: {len(responses_cn)} OK, {len(errores)} errores")

Descargando datos de 34 departamentos...
  [ 1/34] ✅ AMAZONAS (6000)
  [ 2/34] ✅ ANTIOQUIA (0100)
  [ 3/34] ✅ ARAUCA (4000)
  [ 4/34] ✅ ATLANTICO (0300)
  [ 5/34] ✅ BOGOTA D.C. (1600)
  [ 6/34] ✅ BOLIVAR (0500)
  [ 7/34] ✅ BOYACA (0700)
  [ 8/34] ✅ CALDAS (0900)
  [ 9/34] ✅ CAQUETA (4400)
  [10/34] ✅ CASANARE (4600)
  [11/34] ✅ CAUCA (1100)
  [12/34] ✅ CESAR (1200)
  [13/34] ✅ CHOCO (1700)
  [14/34] ✅ CONSULADOS (8800)
  [15/34] ✅ CORDOBA (1300)
  [16/34] ✅ CUNDINAMARCA (1500)
  [17/34] ✅ GUAINIA (5000)
  [18/34] ✅ GUAVIARE (5400)
  [19/34] ✅ HUILA (1900)
  [20/34] ✅ LA GUAJIRA (4800)
  [21/34] ✅ MAGDALENA (2100)
  [22/34] ✅ META (5200)
  [23/34] ✅ NARIÑO (2300)
  [24/34] ✅ NORTE DE SAN (2500)
  [25/34] ✅ PUTUMAYO (6400)
  [26/34] ✅ QUINDIO (2600)
  [27/34] ✅ RISARALDA (2400)
  [28/34] ✅ SAN ANDRES (5600)
  [29/34] ✅ SANTANDER (2700)
  [30/34] ✅ SUCRE (2800)
  [31/34] ✅ TOLIMA (2900)
  [32/34] ✅ VALLE (3100)
  [33/34] ✅ VAUPES (6800)
  [34/34] ✅ VICHADA (7200)

Completado: 34 OK, 0 err

In [38]:
dim_partidos.info()

<class 'pandas.DataFrame'>
RangeIndex: 348 entries, 0 to 347
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   partido_codigo  348 non-null    str  
 1   partido_nombre  348 non-null    str  
 2   partido_color   348 non-null    str  
 3   partido_indice  348 non-null    str  
 4   partido_slug    348 non-null    str  
dtypes: str(5)
memory usage: 39.3 KB


In [41]:
dim_partidos[dim_partidos['partido_indice'] == '30']

,partido_codigo,partido_nombre,partido_color,partido_indice,partido_slug
29,100,"CONSULTA DE LAS SOLUCIONES: SALUD, SEGURIDAD Y...",#407957,30,"CONSULTA-DE-LAS-SOLUCIONES:-SALUD,-SEGURIDAD-Y..."


### Construyendo DataFrames estructurados

De cada response departamental extraemos dos tablas:

1. **`cn_candidatos_depto`**: Votos de cada candidato por departamento (desde `partotabla`)
2. **`cn_municipios_ganador`**: Partido ganador y votos por cada municipio (desde `mapagan`)

In [47]:
# === Tabla 1: cn_candidatos_depto ===
# Votos de cada candidato por partido, por departamento
filas_candidatos = []

for depto_codigo, resp in responses_cn.items():
    depto_nombre = dim_departamentos[
        dim_departamentos['depto_codigo'] == depto_codigo
    ]['depto_nombre'].values[0]
    
    cam = resp['camaras'][0]
    
    for partido in cam['partotabla']:
        p = partido['act']
        codpar = p['codpar']
        votos_partido = p['vot']
        
        for candidato in p['cantotabla']:
            filas_candidatos.append({
                'depto_codigo': depto_codigo,
                'depto_nombre': depto_nombre,
                'partido_codigo': codpar,
                'votos_partido': int(votos_partido),
                'candidato_codigo': candidato['codcan'],
                'candidato_cedula': candidato['cedula'],
                'candidato_nombre': f"{candidato['nomcan']} {candidato['apecan']}".strip(),
                'candidato_votos': int(candidato['vot']),
                'candidato_porcentaje': candidato['pvot'],
                'preferido': candidato.get('pref', '0'),
            })

cn_candidatos_depto = pd.DataFrame(filas_candidatos)

# Enriquecer con nombre del partido desde dim_partidos
cn_candidatos_depto = cn_candidatos_depto.merge(
    dim_partidos[['partido_indice', 'partido_nombre']],
    left_on='partido_codigo',   # Columna en cn_candidatos_depto
    right_on='partido_indice',  # Columna en dim_partidos
    how='left'
)

print(f"cn_candidatos_depto: {len(cn_candidatos_depto)} filas")
print(f"  Departamentos: {cn_candidatos_depto['depto_nombre'].nunique()}")
print(f"  Candidatos únicos: {cn_candidatos_depto['candidato_nombre'].nunique()}")
print(f"  Partidos: {cn_candidatos_depto['partido_nombre'].nunique()}")
cn_candidatos_depto.head(10)

cn_candidatos_depto: 544 filas
  Departamentos: 34
  Candidatos únicos: 16
  Partidos: 3


,depto_codigo,depto_nombre,partido_codigo,votos_partido,candidato_codigo,candidato_cedula,candidato_nombre,candidato_votos,candidato_porcentaje,preferido,partido_indice,partido_nombre
0,6000,AMAZONAS,30,852,1,51992648,CLAUDIA NAYIBE LOPEZ HERNANDEZ,797,"13,41%",1,30,"CONSULTA DE LAS SOLUCIONES: SALUD, SEGURIDAD Y..."
1,6000,AMAZONAS,30,852,2,10004127,LEONARDO HUMBERTO HUERTA GUTIERREZ,55,"0,92%",1,30,"CONSULTA DE LAS SOLUCIONES: SALUD, SEGURIDAD Y..."
2,6000,AMAZONAS,31,4195,5,25280205,PALOMA SUSANA VALENCIA LASERNA,2146,"36,13%",1,31,LA GRAN CONSULTA POR COLOMBIA
3,6000,AMAZONAS,31,4195,9,79941641,JUAN DANIEL OVIEDO ARANGO,913,"15,37%",1,31,LA GRAN CONSULTA POR COLOMBIA
4,6000,AMAZONAS,31,4195,4,79589617,JUAN MANUEL GALAN PACHON,371,"6,24%",1,31,LA GRAN CONSULTA POR COLOMBIA
5,6000,AMAZONAS,31,4195,3,66918051,VICTORIA EUGENIA DAVILA HOYOS,291,"4,89%",1,31,LA GRAN CONSULTA POR COLOMBIA
6,6000,AMAZONAS,31,4195,6,79593847,JUAN CARLOS PINZON BUENO,158,"2,66%",1,31,LA GRAN CONSULTA POR COLOMBIA
7,6000,AMAZONAS,31,4195,8,19333686,ENRIQUE PEÑALOSA LONDOÑO,134,"2,25%",1,31,LA GRAN CONSULTA POR COLOMBIA
8,6000,AMAZONAS,31,4195,2,79777591,DAVID ANDRES LUNA SANCHEZ,78,"1,31%",1,31,LA GRAN CONSULTA POR COLOMBIA
9,6000,AMAZONAS,31,4195,1,79154695,MAURICIO CARDENAS SANTAMARIA,53,"0,89%",1,31,LA GRAN CONSULTA POR COLOMBIA


In [57]:
# === Tabla 2: cn_municipios_ganador ===
# Partido ganador y votos por cada municipio (desde mapagan)
filas_municipios = []

for depto_codigo, resp in responses_cn.items():
    depto_nombre = dim_departamentos[
        dim_departamentos['depto_codigo'] == depto_codigo
    ]['depto_nombre'].values[0]
    
    cam = resp['camaras'][0]
    
    for mpio in cam['mapagan']:
        filas_municipios.append({
            'depto_codigo': depto_codigo,
            'depto_nombre': depto_nombre,
            'mpio_codigo': mpio['amb'],
            'mpio_nombre': mpio.get('nombre', ''),
            'ganador_partido_codigo': mpio['codpar'],
            'ganador_votos': int(mpio['vot']),
            'ganador_porcentaje': mpio['pvot'],
            'total_votantes': int(mpio['votant']),
            'participacion': mpio['pvotant'],
            'votos_validos': int(mpio['votcan']),
            'mesas_escrutadas': int(mpio['mesesc']),
            'pct_mesas_escrutadas': mpio['pmesesc'],
            'hay_empate': mpio.get('hayEmpate', '0'),
        })

cn_municipios_ganador = pd.DataFrame(filas_municipios)

# Enriquecer con nombre del partido ganador
cn_municipios_ganador = cn_municipios_ganador.merge(
    dim_partidos[['partido_indice', 'partido_nombre']], 
    left_on='ganador_partido_codigo',
    right_on='partido_indice',
    how='left'
).drop(columns=['partido_indice']).rename(columns={'partido_nombre': 'ganador_partido_nombre'})

print(f"cn_municipios_ganador: {len(cn_municipios_ganador)} filas")
print(f"  Departamentos: {cn_municipios_ganador['depto_nombre'].nunique()}")
print(f"  Municipios: {cn_municipios_ganador['mpio_nombre'].nunique()}")

cn_municipios_ganador.head(10)

cn_municipios_ganador: 1189 filas
  Departamentos: 34
  Municipios: 1100


,depto_codigo,depto_nombre,mpio_codigo,mpio_nombre,ganador_partido_codigo,ganador_votos,ganador_porcentaje,total_votantes,participacion,votos_validos,mesas_escrutadas,pct_mesas_escrutadas,hay_empate,ganador_partido_nombre
0,6000,AMAZONAS,6000010,EL ENCANTO,31,24,"72,72%",51,"6,62%",33,3,100%,0,LA GRAN CONSULTA POR COLOMBIA
1,6000,AMAZONAS,6000013,LA CHORRERA,31,10,"76,92%",21,"1,81%",13,4,100%,0,LA GRAN CONSULTA POR COLOMBIA
2,6000,AMAZONAS,6000016,LA PEDRERA,31,21,"45,65%",77,"5,78%",46,4,100%,0,LA GRAN CONSULTA POR COLOMBIA
3,6000,AMAZONAS,6000017,LA VICTORIA,31,6,"66,66%",29,"38,15%",9,1,100%,0,LA GRAN CONSULTA POR COLOMBIA
4,6000,AMAZONAS,6000001,LETICIA,31,3948,"71,74%",6713,"14,81%",5503,142,100%,0,LA GRAN CONSULTA POR COLOMBIA
5,6000,AMAZONAS,6000019,MIRITI PARANA,30,3,"37,50%",11,"1,98%",8,3,100%,1,"CONSULTA DE LAS SOLUCIONES: SALUD, SEGURIDAD Y..."
6,6000,AMAZONAS,6000030,PUERTO ALEGRIA,31,4,100%,5,"1,71%",4,1,100%,0,LA GRAN CONSULTA POR COLOMBIA
7,6000,AMAZONAS,6000040,PUERTO ARICA,31,10,"76,92%",15,"2,75%",13,2,100%,0,LA GRAN CONSULTA POR COLOMBIA
8,6000,AMAZONAS,6000007,PUERTO NARIÑO,31,130,"52,84%",374,"7,19%",246,17,100%,0,LA GRAN CONSULTA POR COLOMBIA
9,6000,AMAZONAS,6000021,PUERTO SANTANDER,31,2,"66,66%",9,"1,52%",3,2,100%,0,LA GRAN CONSULTA POR COLOMBIA


### Máxima granularidad: votos por candidato por municipio

Con las 34 consultas departamentales ya tenemos el **ganador** por municipio, pero no los votos de **todos** los candidatos en cada municipio. Para eso necesitamos consultar los 1,189 municipios individualmente.

Usamos `ThreadPoolExecutor` (20 workers) + `requests.Session` con connection pool para paralelizar las descargas.

In [62]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
import threading

# === Scraping paralelo de TODOS los municipios para Consultas (CN) ===
BASE_URL = "https://resultados.registraduria.gov.co/json/ACT/CN"
MAX_WORKERS = 20

# Session con connection pool dimensionado para los workers
session = requests.Session()
session.cookies.update(cookies)
session.headers.update(headers)
adapter = HTTPAdapter(pool_connections=MAX_WORKERS, pool_maxsize=MAX_WORKERS)
session.mount('https://', adapter)

lock = threading.Lock()
completados = 0

def fetch_municipio(row):
    """Descarga y parsea los resultados de un municipio."""
    global completados
    codigo = row['mpio_codigo']
    nombre = row['mpio_nombre']
    depto = row['depto_nombre']
    
    r = session.get(f"{BASE_URL}/{codigo}.json")
    
    with lock:
        completados += 1
        if completados % 200 == 0:
            print(f"  [{completados:4d}/{total}] procesados...")
    
    if r.status_code != 200:
        return [], (codigo, nombre, r.status_code)
    
    resp = r.json()
    cam = resp['camaras'][0]
    totales = cam['totales']['act']
    
    filas = []
    for partido in cam['partotabla']:
        p = partido['act']
        for candidato in p['cantotabla']:
            filas.append({
                'depto_nombre': depto,
                'mpio_codigo': codigo,
                'mpio_nombre': nombre,
                'partido_codigo': p['codpar'],
                'votos_partido_mpio': int(p['vot']),
                'candidato_codigo': candidato['codcan'],
                'candidato_cedula': candidato['cedula'],
                'candidato_nombre': f"{candidato['nomcan']} {candidato['apecan']}".strip(),
                'candidato_votos': int(candidato['vot']),
                'candidato_porcentaje': candidato['pvot'],
                'mpio_votantes': int(totales.get('votant', 0)),
                'mpio_votos_nulos': int(totales.get('votnul', 0)),
                'mpio_votos_no_marcados': int(totales.get('votnma', 0)),
                'mpio_votos_validos': int(totales.get('votval', 0)),
            })
    return filas, None

rows = [row for _, row in dim_municipios.iterrows()]
total = len(rows)
filas_cn_mpio = []
errores_mpio = []

print(f"Descargando datos de {total} municipios ({MAX_WORKERS} workers concurrentes)...")
t0 = time.time()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(fetch_municipio, row): row for row in rows}
    for future in as_completed(futures):
        filas, error = future.result()
        filas_cn_mpio.extend(filas)
        if error:
            errores_mpio.append(error)

elapsed = time.time() - t0
print(f"\n✅ Completado en {elapsed:.1f}s: {total - len(errores_mpio)} OK, {len(errores_mpio)} errores")
print(f"   Filas extraídas: {len(filas_cn_mpio)}")
if errores_mpio:
    print(f"   Errores: {errores_mpio[:5]}")

Descargando datos de 1189 municipios (20 workers concurrentes)...
  [ 200/1189] procesados...
  [ 400/1189] procesados...
  [ 600/1189] procesados...
  [ 800/1189] procesados...
  [1000/1189] procesados...

✅ Completado en 15.1s: 1189 OK, 0 errores
   Filas extraídas: 19024


In [63]:
# === DataFrame final: cn_candidatos_municipio ===
cn_candidatos_municipio = pd.DataFrame(filas_cn_mpio)

# Enriquecer con nombre del partido
cn_candidatos_municipio = cn_candidatos_municipio.merge(
    dim_partidos[['partido_indice', 'partido_nombre']],
    left_on='partido_codigo',   # Columna en cn_candidatos_municipio
    right_on='partido_indice',  # Columna en dim_partidos
    how='left'
).drop(columns=['partido_indice'])

print(f"cn_candidatos_municipio: {len(cn_candidatos_municipio)} filas")
print(f"  Departamentos: {cn_candidatos_municipio['depto_nombre'].nunique()}")
print(f"  Municipios: {cn_candidatos_municipio['mpio_nombre'].nunique()}")
print(f"  Candidatos: {cn_candidatos_municipio['candidato_nombre'].nunique()}")
print(f"  Partidos: {cn_candidatos_municipio['partido_nombre'].nunique()}")
print(f"\nColumnas: {list(cn_candidatos_municipio.columns)}")
cn_candidatos_municipio.head(10)

cn_candidatos_municipio: 19024 filas
  Departamentos: 34
  Municipios: 1100
  Candidatos: 16
  Partidos: 3

Columnas: ['depto_nombre', 'mpio_codigo', 'mpio_nombre', 'partido_codigo', 'votos_partido_mpio', 'candidato_codigo', 'candidato_cedula', 'candidato_nombre', 'candidato_votos', 'candidato_porcentaje', 'mpio_votantes', 'mpio_votos_nulos', 'mpio_votos_no_marcados', 'mpio_votos_validos', 'partido_nombre']


,depto_nombre,mpio_codigo,mpio_nombre,partido_codigo,votos_partido_mpio,candidato_codigo,candidato_cedula,candidato_nombre,candidato_votos,candidato_porcentaje,mpio_votantes,mpio_votos_nulos,mpio_votos_no_marcados,mpio_votos_validos,partido_nombre
0,HUILA,1900007,AGRADO,30,50,1,51992648,CLAUDIA NAYIBE LOPEZ HERNANDEZ,44,"4,88%",1137,107,130,900,"CONSULTA DE LAS SOLUCIONES: SALUD, SEGURIDAD Y..."
1,HUILA,1900007,AGRADO,30,50,2,10004127,LEONARDO HUMBERTO HUERTA GUTIERREZ,6,"0,66%",1137,107,130,900,"CONSULTA DE LAS SOLUCIONES: SALUD, SEGURIDAD Y..."
2,HUILA,1900007,AGRADO,31,772,5,25280205,PALOMA SUSANA VALENCIA LASERNA,520,"57,77%",1137,107,130,900,LA GRAN CONSULTA POR COLOMBIA
3,HUILA,1900007,AGRADO,31,772,9,79941641,JUAN DANIEL OVIEDO ARANGO,114,"12,66%",1137,107,130,900,LA GRAN CONSULTA POR COLOMBIA
4,HUILA,1900007,AGRADO,31,772,3,66918051,VICTORIA EUGENIA DAVILA HOYOS,48,"5,33%",1137,107,130,900,LA GRAN CONSULTA POR COLOMBIA
5,HUILA,1900007,AGRADO,31,772,4,79589617,JUAN MANUEL GALAN PACHON,45,"5,00%",1137,107,130,900,LA GRAN CONSULTA POR COLOMBIA
6,HUILA,1900007,AGRADO,31,772,6,79593847,JUAN CARLOS PINZON BUENO,20,"2,22%",1137,107,130,900,LA GRAN CONSULTA POR COLOMBIA
7,HUILA,1900007,AGRADO,31,772,8,19333686,ENRIQUE PEÑALOSA LONDOÑO,10,"1,11%",1137,107,130,900,LA GRAN CONSULTA POR COLOMBIA
8,HUILA,1900007,AGRADO,31,772,2,79777591,DAVID ANDRES LUNA SANCHEZ,9,"1,00%",1137,107,130,900,LA GRAN CONSULTA POR COLOMBIA
9,HUILA,1900007,AGRADO,31,772,1,79154695,MAURICIO CARDENAS SANTAMARIA,3,"0,33%",1137,107,130,900,LA GRAN CONSULTA POR COLOMBIA


In [67]:
# === Resumen: votos totales por candidato a nivel nacional ===
resumen = cn_candidatos_municipio.groupby(
    ['partido_nombre', 'candidato_nombre']
).agg(
    votos_totales=('candidato_votos', 'sum'),
    municipios_presencia=('mpio_codigo', 'nunique'),
).sort_values('votos_totales', ascending=False).reset_index()

print("Votos totales por candidato (Consultas Interpartidistas - nivel municipio):")
print("=" * 90)
resumen

Votos totales por candidato (Consultas Interpartidistas - nivel municipio):


,partido_nombre,candidato_nombre,votos_totales,municipios_presencia
0,LA GRAN CONSULTA POR COLOMBIA,PALOMA SUSANA VALENCIA LASERNA,3236286,1189
1,LA GRAN CONSULTA POR COLOMBIA,JUAN DANIEL OVIEDO ARANGO,1255510,1189
2,"CONSULTA DE LAS SOLUCIONES: SALUD, SEGURIDAD Y...",CLAUDIA NAYIBE LOPEZ HERNANDEZ,574670,1189
3,LA GRAN CONSULTA POR COLOMBIA,JUAN MANUEL GALAN PACHON,327765,1189
4,LA GRAN CONSULTA POR COLOMBIA,JUAN CARLOS PINZON BUENO,297646,1189
5,FRENTE POR LA VIDA,ROY LEONARDO BARRERAS MONTEALEGRE,257037,1189
6,LA GRAN CONSULTA POR COLOMBIA,VICTORIA EUGENIA DAVILA HOYOS,238045,1189
7,FRENTE POR LA VIDA,DANIEL QUINTERO CALLE,227379,1189
8,LA GRAN CONSULTA POR COLOMBIA,ENRIQUE PEÑALOSA LONDOÑO,162534,1189
9,LA GRAN CONSULTA POR COLOMBIA,ANIBAL GAVIRIA CORREA,149091,1189


## Exportación de datos

Disponibilizamos el DataFrame más granular (`cn_candidatos_municipio`) en múltiples formatos dentro de `data/raw_data_extract/consultas_interpartidistas/`:

| Formato | Archivo | Uso típico |
|---------|---------|------------|
| CSV (coma) | `.csv` | Compatible universal, Excel, R, Python |
| CSV (punto y coma) | `_semicolon.csv` | Excel en configuraciones regionales con coma decimal |
| Excel | `.xlsx` | Análisis rápido en hojas de cálculo |
| JSON | `.json` | APIs, JavaScript, visualizaciones web |
| Parquet | `.parquet` | Alto rendimiento para grandes volúmenes, Spark, pandas |

In [68]:
import os

# Directorio de salida (relativo a la raíz del proyecto)
OUTPUT_DIR = os.path.join('..', '..', 'data', 'raw_data_extract', 'consultas_interpartidistas')
os.makedirs(OUTPUT_DIR, exist_ok=True)

base = os.path.join(OUTPUT_DIR, 'cn_candidatos_municipio')

# CSV (separado por coma)
cn_candidatos_municipio.to_csv(f'{base}.csv', index=False, encoding='utf-8-sig')

# CSV (separado por punto y coma)
cn_candidatos_municipio.to_csv(f'{base}_semicolon.csv', index=False, encoding='utf-8-sig', sep=';')

# Excel
cn_candidatos_municipio.to_excel(f'{base}.xlsx', index=False, engine='openpyxl')

# JSON
cn_candidatos_municipio.to_json(f'{base}.json', orient='records', force_ascii=False, indent=2)

# Parquet
cn_candidatos_municipio.to_parquet(f'{base}.parquet', index=False, engine='pyarrow')

print(f"✅ Exportados 5 archivos en {OUTPUT_DIR}/")
for ext in ['.csv', '_semicolon.csv', '.xlsx', '.json', '.parquet']:
    filepath = f'{base}{ext}'
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f"   {os.path.basename(filepath):45s} {size_mb:.2f} MB")

✅ Exportados 5 archivos en ../../data/raw_data_extract/consultas_interpartidistas/
   cn_candidatos_municipio.csv                   2.37 MB
   cn_candidatos_municipio_semicolon.csv         2.33 MB
   cn_candidatos_municipio.xlsx                  1.15 MB
   cn_candidatos_municipio.json                  9.19 MB
   cn_candidatos_municipio.parquet               0.17 MB
